In [ ]:
!python --version

In [ ]:
# This is for colab users to mount google drive
#from google.colab import drive
#drive.mount('/content/gdrive')

In [ ]:
# install pytorch (http://pytorch.org/) if run from Google Colaboratory
import sys
%matplotlib inline
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

In [ ]:
# 2-d latent space, parameter count in the same order of magnitude
# as in the original VAE paper (VAE paper has about 3x as many)
latent_dims = 2
num_epochs = 100
batch_size = 128
capacity = 64
learning_rate = 1e-3
variational_beta = 1
use_gpu = True
savepath='vae_2d_100.pth'

# # 10-d latent space, for comparison with non-variational auto-encoder
# latent_dims = 10
# num_epochs = 100
# batch_size = 128
# capacity = 64
# learning_rate = 1e-3
# variational_beta = 1
# use_gpu = True

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST

img_transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = MNIST(root='./data/MNIST', download=True, train=True, transform=img_transform)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = MNIST(root='./data/MNIST', download=True, train=False, transform=img_transform)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
from torchvision.transforms.functional import to_pil_image
idx = 42
print(train_dataset[idx][1]) # class
to_pil_image(train_dataset[idx][0]) # image data

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        c = capacity
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=c, kernel_size=4, stride=2, padding=1) # out: c x 14 x 14
        self.conv2 = nn.Conv2d(in_channels=c, out_channels=c*2, kernel_size=4, stride=2, padding=1) # out: 2c x 7 x 7
        self.fc_mu = nn.Linear(in_features=c*2*7*7, out_features=latent_dims)
        self.fc_logvar = nn.Linear(in_features=c*2*7*7, out_features=latent_dims)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1) # flatten batch of multi-channel feature maps to a batch of feature vectors
        x_mu = self.fc_mu(x)
        x_logvar = self.fc_logvar(x)
        return x_mu, x_logvar


In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        c = capacity
        self.fc = nn.Linear(in_features=latent_dims, out_features=c*2*7*7)
        self.conv2 = nn.ConvTranspose2d(in_channels=c*2, out_channels=c, kernel_size=4, stride=2, padding=1)
        self.conv1 = nn.ConvTranspose2d(in_channels=c, out_channels=1, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), capacity*2, 7, 7) # unflatten batch of feature vectors to a batch of multi-channel feature maps
        x = F.relu(self.conv2(x))
        x = torch.sigmoid(self.conv1(x)) # last layer before output is sigmoid, since we are using BCE as reconstruction loss
        return x

In [ ]:
class VariationalAutoencoder(nn.Module):
    def __init__(self):
        super(VariationalAutoencoder, self).__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        latent_mu, latent_logvar = self.encoder(x)
        latent = self.latent_sample(latent_mu, latent_logvar)
        x_recon = self.decoder(latent)
        return x_recon, latent_mu, latent_logvar

    def latent_sample(self, mu, logvar):
        if self.training:
            # the reparameterization trick
            std = logvar.mul(0.5).exp_()
            eps = torch.empty_like(std).normal_()
            return eps.mul(std).add_(mu)
        else:
            return mu

In [ ]:
vae = VariationalAutoencoder()

device = torch.device("cuda:0" if use_gpu and torch.cuda.is_available() else "cpu")
vae = vae.to(device)

num_params = sum(p.numel() for p in vae.parameters() if p.requires_grad)
print('Number of parameters: %d' % num_params)

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    # recon_x is the probability of a multivariate Bernoulli distribution p.
    # -log(p(x)) is then the pixel-wise binary cross-entropy.
    # Averaging or not averaging the binary cross-entropy over all pixels here
    # is a subtle detail with big effect on training, since it changes the weight
    # we need to pick for the other loss term by several orders of magnitude.
    # Not averaging is the direct implementation of the negative log likelihood,
    # but averaging makes the weight of the other loss term independent of the image resolution.
    recon_loss = F.binary_cross_entropy(recon_x.view(-1, 784), x.view(-1, 784), reduction='sum')

    # KL-divergence between the prior distribution over latent vectors
    # (the one we are going to sample from when generating new images)
    # and the distribution estimated by the generator for the given image.
    kldivergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return recon_loss + variational_beta * kldivergence

In [ ]:
optimizer = torch.optim.Adam(params=vae.parameters(), lr=learning_rate, weight_decay=1e-5)

if os.path.isfile(savepath):
    print(f'Loading pretrained VAE from {savepath}')
    vae.load_state_dict(torch.load(savepath))
    vae.eval()
else:
    # set to training mode
    vae.train()

    train_loss_avg = []

    print('Training ...')
    for epoch in range(num_epochs):
        train_loss_avg.append(0)
        num_batches = 0

        for image_batch, _ in train_dataloader:

            image_batch = image_batch.to(device)

            # vae reconstruction
            image_batch_recon, latent_mu, latent_logvar = vae(image_batch)

            # reconstruction error
            loss = vae_loss(image_batch_recon, image_batch, latent_mu, latent_logvar)

            # backpropagation
            optimizer.zero_grad()
            loss.backward()

            # one step of the optmizer (using the gradients from backpropagation)
            optimizer.step()

            train_loss_avg[-1] += loss.item()
            num_batches += 1

        train_loss_avg[-1] /= num_batches
        print('Epoch [%d / %d] average reconstruction error: %f' % (epoch+1, num_epochs, train_loss_avg[-1]))

In [ ]:
if not os.path.isfile(savepath):
    torch.save(vae.state_dict(), savepath)

In [ ]:
import matplotlib.pyplot as plt
plt.ion()

if not os.path.isfile(savepath):
    fig = plt.figure()
    plt.plot(train_loss_avg)
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.show()

In [ ]:
# set to evaluation mode
vae.eval()

test_loss_avg, num_batches = 0, 0
for image_batch, _ in test_dataloader:

    with torch.no_grad():

        image_batch = image_batch.to(device)

        # vae reconstruction
        image_batch_recon, latent_mu, latent_logvar = vae(image_batch)

        # reconstruction error
        loss = vae_loss(image_batch_recon, image_batch, latent_mu, latent_logvar)

        test_loss_avg += loss.item()
        num_batches += 1

test_loss_avg /= num_batches
print('average reconstruction error: %f' % (test_loss_avg))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.ion()

import torchvision.utils

vae.eval()

# This function takes as an input the images to reconstruct
# and the name of the model with which the reconstructions
# are performed
def to_img(x):
    x = x.clamp(0, 1)
    return x

def show_image(img):
    img = to_img(img)
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

def visualise_output(images, model):

    with torch.no_grad():

        images = images.to(device)
        images, _, _ = model(images)
        images = images.cpu()
        images = to_img(images)
        np_imagegrid = torchvision.utils.make_grid(images[1:50], 10, 5).numpy()
        plt.imshow(np.transpose(np_imagegrid, (1, 2, 0)))
        plt.show()

images, labels = next(iter(test_dataloader)) # Bug Fix

# First visualise the original images
print('Original images')
show_image(torchvision.utils.make_grid(images[1:50],10,5))
plt.show()

# Reconstruct and visualise the images using the vae
print('VAE reconstruction:')
visualise_output(images, vae)

In [ ]:
vae.eval()
def interpolation(lambda1, model, img1, img2):

    with torch.no_grad():

        # latent vector of first image
        img1 = img1.to(device)
        latent_1, _ = model.encoder(img1)

        # latent vector of second image
        img2 = img2.to(device)
        latent_2, _ = model.encoder(img2)

        # interpolation of the two latent vectors
        inter_latent = lambda1* latent_1 + (1- lambda1) * latent_2

        # reconstruct interpolated image
        inter_image = model.decoder(inter_latent)
        inter_image = inter_image.cpu()

        return inter_image

# sort part of test set by digit
digits = [[] for _ in range(10)]
for img_batch, label_batch in test_dataloader:
    for i in range(img_batch.size(0)):
        digits[label_batch[i]].append(img_batch[i:i+1])
    if sum(len(d) for d in digits) >= 1000:
        break;

# interpolation lambdas
lambda_range=np.linspace(0,1,10)

fig, axs = plt.subplots(2,5, figsize=(15, 6))
fig.subplots_adjust(hspace = .5, wspace=.001)
axs = axs.ravel()

for ind,l in enumerate(lambda_range):
    inter_image=interpolation(1-float(l), vae, digits[0][0], digits[9][0])

    inter_image = to_img(inter_image)

    image = inter_image.numpy()

    axs[ind].imshow(image[0,0,:,:], cmap='gray')
    axs[ind].set_title('lambda_val='+str(round(l,1)))
plt.show()

In [ ]:
vae.eval()

with torch.no_grad():

    # sample latent vectors from the normal distribution
    latent = torch.randn(128, latent_dims, device=device)

    # reconstruct images from the latent vectors
    img_recon = vae.decoder(latent)
    img_recon = img_recon.cpu()

    fig, ax = plt.subplots(figsize=(5, 5))
    show_image(torchvision.utils.make_grid(img_recon.data[:100],10,5))
    plt.show()

In [ ]:
# Answer 1
img4 = digits[4][5]
img8 = digits[8][5]

lambda_range=np.linspace(0,1,10)

fig, axs = plt.subplots(2, 5, figsize=(15, 6))
fig.subplots_adjust(hspace = .5, wspace=.001)
axs = axs.ravel()

for ind, l in enumerate(lambda_range):
    inter_image = (1 - l) * img4 + l * img8

    inter_image = to_img(inter_image)

    image = inter_image.numpy()

    axs[ind].imshow(image[0,0,:,:], cmap='gray')
    axs[ind].set_title('lambda_val='+str(round(l,1)))
plt.show()

In [ ]:
# Answer 2
vae.eval()
lambda_range=np.linspace(0,1,10)

fig, axs = plt.subplots(2, 5, figsize=(15, 6))
fig.subplots_adjust(hspace = .5, wspace=.001)
axs = axs.ravel()

for ind, l in enumerate(lambda_range):
    inter_image=interpolation(1-float(l), vae, img4, img8)

    inter_image = to_img(inter_image)

    image = inter_image.numpy()

    axs[ind].imshow(image[0,0,:,:], cmap='gray')
    axs[ind].set_title('lambda_val='+str(round(l,1)))
plt.show()

In [ ]:
# Answer 3
vae.eval()

imgs7 = torch.cat(digits[7], dim=0).to(device)

mean_x = imgs7.mean(dim=0)
median_x = imgs7.median(dim=0).values

latents = []
with torch.no_grad():
    for img7 in digits[7]:
        img7 = img7.to(device)
        latent_mu, latent_logvar = vae.encoder(img7)
        latent = vae.latent_sample(latent_mu, latent_logvar)
        latents.append(latent.squeeze(0))
latents = torch.stack(latents)

mean_z = latents.mean(dim=0)
median_z = latents.median(dim=0).values

decoded_mean_z = vae.decoder(mean_z.unsqueeze(0))
decoded_median_z = vae.decoder(median_z.unsqueeze(0))

fig, axs = plt.subplots(2, 2, figsize=(6, 6))
fig.subplots_adjust(hspace = .5, wspace=.001)
axs = axs.ravel()

axs[0].imshow(mean_x.squeeze().cpu().numpy(), cmap='gray')
axs[0].set_title('Mean of 7 in input space')
axs[1].imshow(median_x.squeeze().cpu().numpy(), cmap='gray')
axs[1].set_title('Median of 7 in input space')
axs[2].imshow(decoded_mean_z.squeeze().cpu().detach().numpy(), cmap='gray')
axs[2].set_title('Mean of 7 in latent space')
axs[3].imshow(decoded_median_z.squeeze().cpu().detach().numpy(), cmap='gray')
axs[3].set_title('Median of 7 in latent space')
plt.show()

In [ ]:
# Answer 4
vae.eval()

diff_x = imgs7 - mean_x
dists_x = torch.norm(diff_x.view(diff_x.size(0), -1), dim=1)
farthest_x = imgs7[dists_x.argmax()]

dists_z = torch.norm(latents - mean_z, dim=1)
farthest_z = latents[dists_z.argmax()]
decoded_farthest_z = vae.decoder(farthest_z.unsqueeze(0))

fig, axs = plt.subplots(1, 2, figsize=(6, 6))
fig.subplots_adjust(hspace = .5, wspace=.3)
axs = axs.ravel()

axs[0].imshow(farthest_x.squeeze().cpu().numpy(), cmap='gray')
axs[0].set_title('Most distant 7 in input space')
axs[1].imshow(decoded_farthest_z.squeeze().cpu().detach().numpy(), cmap='gray')
axs[1].set_title('Most distant 7 in latent space')
plt.show()

In [ ]:
# Answer 5
vae.eval()

all_imgs = []
all_latents = []

with torch.no_grad():
    for img_batch, _ in test_dataloader:
        img_batch = img_batch.to(device)
        latent_mu, latent_logvar = vae.encoder(img_batch)
        latent = vae.latent_sample(latent_mu, latent_logvar)
        all_imgs.append(img_batch)
        all_latents.append(latent)

all_imgs = torch.cat(all_imgs, dim=0)
median_x = all_imgs.median(dim=0).values

all_latents = torch.cat(all_latents, dim=0)
median_z = all_latents.median(dim=0).values
decoded_median_z = vae.decoder(median_z.unsqueeze(0))

fig, axs = plt.subplots(1, 2, figsize=(6, 6))
fig.subplots_adjust(hspace = .5, wspace=.7)
axs = axs.ravel()

axs[0].imshow(median_x.squeeze().cpu().numpy(), cmap='gray')
axs[0].set_title('Median of all digits in input space')
axs[1].imshow(decoded_median_z.squeeze().cpu().detach().numpy(), cmap='gray')
axs[1].set_title('Median of all digits in latent space')
plt.show()

In [ ]:
# Answer 6
train_losses_savepath = 'train_losses.pt'
test_losses_savepath = 'test_losses.pt'

variational_betas = [0.1, 0.5, 1.0, 2.0, 5.0]
vaes = []
train_losses = []

def vae_loss_beta(recon_x, x, mu, logvar, beta=1):
    recon_loss = F.binary_cross_entropy(recon_x.view(-1, 784), x.view(-1, 784), reduction='sum')
    kldivergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kldivergence

for beta in variational_betas:
    vae_savepath = f'vae_2d_10_beta{beta}.pth'

    vae = VariationalAutoencoder().to(device)
    optimizer = torch.optim.Adam(params=vae.parameters(), lr=learning_rate, weight_decay=1e-5)

    if os.path.isfile(vae_savepath) and os.path.isfile(train_losses_savepath):
        print(f'Loading pretrained VAE from {vae_savepath}')
        vae.load_state_dict(torch.load(vae_savepath))
    else:
        vae.train()

        train_loss_avg = []

        print(f'\nTraining VAE with beta = {beta}')

        for epoch in range(10):
            train_loss = 0

            for image_batch, _ in train_dataloader:
                image_batch = image_batch.to(device)

                # vae reconstruction
                image_batch_recon, latent_mu, latent_logvar = vae(image_batch)

                # reconstruction error
                loss = vae_loss_beta(image_batch_recon, image_batch, latent_mu, latent_logvar, beta)

                # backpropagation
                optimizer.zero_grad()
                loss.backward()

                # one step of the optmizer (using the gradients from backpropagation)
                optimizer.step()

                train_loss += loss.item()

            train_loss /= len(train_dataloader)
            print('Epoch [%d / %d] average reconstruction error: %f' % (epoch+1, 10, train_loss))

        train_losses.append(train_loss)
        torch.save(vae.state_dict(), vae_savepath)

    vaes.append(vae)

if os.path.isfile(train_losses_savepath):
    train_losses = torch.load(train_losses_savepath)
else:
    torch.save(train_losses, train_losses_savepath)

In [ ]:
test_losses = []

for vae, beta in zip(vaes, variational_betas):
    vae.eval()

    if not os.path.isfile(test_losses_savepath):
        test_loss = 0
        with torch.no_grad():
            for image_batch, _ in test_dataloader:
                image_batch = image_batch.to(device)
                image_batch_recon, latent_mu, latent_logvar = vae(image_batch)
                loss = vae_loss_beta(image_batch_recon, image_batch, latent_mu, latent_logvar, beta)
                test_loss += loss.item()
        test_loss /= len(test_dataloader)
        test_losses.append(test_loss)
        print(f'Test loss for beta = {beta}: {test_loss}')

if os.path.isfile(test_losses_savepath):
    test_losses = torch.load(test_losses_savepath)
else:
    torch.save(test_losses, test_losses_savepath)

In [ ]:
train_losses

In [ ]:
test_losses

In [ ]:
x = np.arange(len(variational_betas))  # [0, 1, 2, 3, 4]
width = 0.35               # Width of a bar

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, train_losses, width, label='Train Loss', color='skyblue')
plt.bar(x + width/2, test_losses, width, label='Test Loss', color='orange')

plt.xticks(x, [str(b) for b in variational_betas])
plt.xlabel('Beta (Regularization Strength)')
plt.ylabel('Average Reconstruction Loss')
plt.title('Reconstruction Loss vs Beta')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
set_seed()

sample_images = []

with torch.no_grad():
    for vae in vaes:
        latent = torch.randn(64, latent_dims).to(device)
        samples = vae.decoder(latent).cpu()
        sample_images.append(samples)

fig, axs = plt.subplots(1, 5, figsize=(12, 3))
axs = axs.ravel()

for i, (samples, beta) in enumerate(zip(sample_images, variational_betas)):
    recon_grid = torchvision.utils.make_grid(samples[:64], nrow=8, padding=1)
    axs[i].imshow(recon_grid.permute(1, 2, 0).numpy(), cmap='gray')
    axs[i].set_title(f"Beta = {beta}")
    axs[i].axis('off')

plt.suptitle("Generated Samples from Latent Prior")
plt.show()